In [2]:
# Environment Setup & Dependency Installation

import sys
import os
import shutil
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

# Detect Environment
def in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IS_COLAB = in_colab()
PYTHON_VERSION = sys.version.split()[0]

print(f"Environment Detected: {'Google Colab' if IS_COLAB else 'Local VS Code / Jupyter'}")
print(f"Kernel Python Version: {PYTHON_VERSION}\n")

# Packages to install
# Ensure PEFT is included for Model C
PACKAGES = "transformers datasets evaluate nltk seaborn matplotlib accelerate numpy pandas torch scikit-learn peft" 

if IS_COLAB:
    # --- Colab-Specific Installation ---
    try:
        import pyarrow
        import datasets
        import evaluate
        import sklearn
        NEEDS_INSTALL = False
    except (ImportError, ValueError):
        NEEDS_INSTALL = True

    if NEEDS_INSTALL:
        print("Installing/Updating core dependencies...")
        !pip install -q pyarrow
        !pip install -q {PACKAGES}
        print("✓ Dependencies installed/updated!")
        print("⚠️  Restarting Runtime...")
        print("📝 Re-run the Notebook...")
        time.sleep(2)
        os.kill(os.getpid(), 9)
    else:
        print("✓ Dependencies already installed!")
else:
    # --- Local Jupyter/VS Code Installation ---\n",
    print("Installing/Updating core dependencies")
    try:
        if 'get_ipython' in globals():
            cell_content = f"%%capture\n%pip install -U {PACKAGES}"
            get_ipython().run_cell(cell_content)
            print("✓ Dependencies installed/updated!")
        else:
            print("⚠️  Installation failed: Not running in interactive environment. Run manually.")
    except NameError:
        print("⚠️  Installation failed: Not running in a standard Jupyter kernel. Run manually.")

Environment Detected: Local VS Code / Jupyter
Kernel Python Version: 3.12.10

Installing/Updating core dependencies
✓ Dependencies installed/updated!


In [3]:
# Importing Libraries

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from datasets import load_from_disk
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# New: Import PEFT modules for Model C (LoRA)
from peft import LoraConfig, get_peft_model, TaskType

# Only needed for Colab-specific functions
if IS_COLAB:
    from IPython.display import display, HTML
    import shutil  # For zipping files
    from google.colab import files  # For downloading files

print("✅ All Libraries Imported Successfully")

c:\Users\joaqu\AI-News-Classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All Libraries Imported Successfully


In [4]:
# Setup Training Paths and Configuration

print("\n Setting up training environment...\n")

# Core constants
MODEL_CKPT = "distilbert-base-uncased"
CLASS_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
model_folder_name = "Distilbert"  # Consistent capitalization

# --- Define Base Paths ---
if IS_COLAB:
    DATASET_PATH = "/content/data"
    MODEL_BASE_DIR = f"/content/models/{model_folder_name}"
else:
    # Local paths - notebook is in notebooks/ folder
    DATASET_PATH = "../data"
    MODEL_BASE_DIR = f"../models/{model_folder_name}"

# --- Define Experiment-Specific Paths (B & C) ---
# UPDATED NAMING CONVENTION HERE: distilbert-B-checkpoints, etc.
OUTPUT_DIR_B = f"{MODEL_BASE_DIR}/distilbert-B-checkpoints"
FINAL_MODEL_DIR_B = f"{MODEL_BASE_DIR}/distilbert-B-final"
OUTPUT_DIR_C = f"{MODEL_BASE_DIR}/distilbert-C-checkpoints"
FINAL_MODEL_DIR_C = f"{MODEL_BASE_DIR}/distilbert-C-final"

# Create directories
os.makedirs(MODEL_BASE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR_B, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR_B, exist_ok=True)
os.makedirs(OUTPUT_DIR_C, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR_C, exist_ok=True)

# Verify dataset exists
if not os.path.exists(DATASET_PATH):
    print(f"Current directory: {os.getcwd()}")
    print(f"Looking for data at: {os.path.abspath(DATASET_PATH)}")
    raise FileNotFoundError(f"❌ Dataset not found at: {DATASET_PATH}\n   Please run Notebook 1 (EDA & Preprocessing) first.")

print(f"✅ Dataset path: {os.path.abspath(DATASET_PATH)}")
print(f"✅ Model base dir: {os.path.abspath(MODEL_BASE_DIR)}")
print(f"✅ Checkpoints B dir: {os.path.abspath(OUTPUT_DIR_B)}")
print(f"✅ Final model B dir: {os.path.abspath(FINAL_MODEL_DIR_B)}")
print(f"✅ Checkpoints C dir: {os.path.abspath(OUTPUT_DIR_C)}")
print(f"✅ Final model C dir: {os.path.abspath(FINAL_MODEL_DIR_C)}\n")


 Setting up training environment...

✅ Dataset path: c:\Users\joaqu\AI-News-Classification\data
✅ Model base dir: c:\Users\joaqu\AI-News-Classification\models\Distilbert
✅ Checkpoints B dir: c:\Users\joaqu\AI-News-Classification\models\Distilbert\distilbert-B-checkpoints
✅ Final model B dir: c:\Users\joaqu\AI-News-Classification\models\Distilbert\distilbert-B-final
✅ Checkpoints C dir: c:\Users\joaqu\AI-News-Classification\models\Distilbert\distilbert-C-checkpoints
✅ Final model C dir: c:\Users\joaqu\AI-News-Classification\models\Distilbert\distilbert-C-final



In [5]:
# Load Dataset

print("📂 Loading tokenized dataset...\n")

tokenized = load_from_disk(DATASET_PATH)
tokenized_train = tokenized["train"]
tokenized_test = tokenized["test"]

print("✅ Dataset Loaded Successfully")
print(f"Train samples: {len(tokenized_train):,}")
print(f"Test samples:  {len(tokenized_test):,}")
print(f"Features: {list(tokenized_train.features.keys())}")

📂 Loading tokenized dataset...

✅ Dataset Loaded Successfully
Train samples: 2,000
Test samples:  500
Features: ['text', 'label', 'input_ids', 'attention_mask']


In [6]:
# ==========================================================
# EXPERIMENT SETUP: MODEL DEFINITION AND TRAINER CONFIG
# ==========================================================

# --- Shared Tokenizer ---
print("🤖 Loading common tokenizer components...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT) 
print("✅ Tokenizer loaded.\n")

# --- Shared Metrics Setup ---
print("📈 Setting up evaluation metrics...")
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    """Compute accuracy and F1 score during evaluation"""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }
print("✅ Metrics configured: Accuracy & F1 (macro)\n")


# --- 1. Model A: ZERO-SHOT BASELINE ---
print("--- 1. Setting up Model A (Zero-Shot Baseline) ---\n")
model_a = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(CLASS_NAMES),
    id2label=CLASS_NAMES,
    label2id={v: k for k, v in CLASS_NAMES.items()}
)
print(f"✅ Model A (Untrained) loaded. Trainable Params: {model_a.num_parameters()}")


# --- 2. Model B: PERFORMANCE BASELINE (FULL FINE-TUNING) ---
print("\n--- 2. Setting up Model B (Standard Full FT) ---\n")
model_b = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(CLASS_NAMES),
    id2label=CLASS_NAMES,
    label2id={v: k for k, v in CLASS_NAMES.items()}
)
print(f"✅ Model B (Full FT) loaded. Trainable Params: {model_b.num_parameters()}")


# --- 3. Model C: EFFICIENCY BASELINE (LoRA/PEFT) ---
print("\n--- 3. Setting up Model C (LoRA/PEFT) ---\n")
model_c_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(CLASS_NAMES),
    id2label=CLASS_NAMES,
    label2id={v: k for k, v in CLASS_NAMES.items()}
)

# Define LoRA Configuration
lora_config = LoraConfig(
    r=8,                     
    lora_alpha=16,           
    target_modules=["q_lin", "v_lin"], 
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS 
)

# Wrap the model with LoRA
model_c = get_peft_model(model_c_base, lora_config)

print("LoRA Parameters:")
model_c.print_trainable_parameters() 
print(f"✅ Model C (LoRA) configured. Checkpoints will be saved to: {OUTPUT_DIR_C}")


# --- TRAINING ARGUMENTS (SHARED) ---
print("\n⚙️  Configuring common Training Arguments...")

# Define baseline training arguments (used for Model B and C)
training_args = TrainingArguments(
    # General Settings
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    
    # Evaluation & Saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=1, 
    report_to="none",
    
    output_dir=OUTPUT_DIR_B, # Placeholder, will be explicitly set in execution
    logging_steps=20,
)

print("✅ Training Arguments defined.")

🤖 Loading common tokenizer components...
✅ Tokenizer loaded.

📈 Setting up evaluation metrics...
✅ Metrics configured: Accuracy & F1 (macro)

--- 1. Setting up Model A (Zero-Shot Baseline) ---



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model A (Untrained) loaded. Trainable Params: 66956548

--- 2. Setting up Model B (Standard Full FT) ---

✅ Model B (Full FT) loaded. Trainable Params: 66956548

--- 3. Setting up Model C (LoRA/PEFT) ---



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LoRA Parameters:
trainable params: 741,124 || all params: 67,697,672 || trainable%: 1.0948
✅ Model C (LoRA) configured. Checkpoints will be saved to: ../models/Distilbert/distilbert-C-checkpoints

⚙️  Configuring common Training Arguments...
✅ Training Arguments defined.


In [ ]:
# ==========================================================
# EXECUTE EXPERIMENTS & SAVE RESULTS
# ==========================================================

import time 
import json
from transformers import TrainingArguments, Trainer 

# Dictionary to store final metrics for comparison
all_metrics = {}
EXPERIMENT_LOG = []
start_time = time.time()

# Get the base training arguments dictionary for easy modification
base_args_dict = training_args.to_dict()

# --- EXPERIMENT 1: MODEL A (ZERO-SHOT BASELINE) ---

print("\n" + "="*60)
print("1. EVALUATING MODEL A (ZERO-SHOT BASELINE)")
print("="*60)

# Trainer A uses the unmodified training_args for evaluation only
trainer_a = Trainer(
    model=model_a,
    args=training_args,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

metrics_a = trainer_a.evaluate()
print(f"\nFinal Test Results (Model A): {metrics_a}")
EXPERIMENT_LOG.append({
    "Model": "Model A",
    "Method": "Zero-Shot Baseline",
    "Trainable Parameters": model_a.num_parameters(),
    "Accuracy": metrics_a.get("eval_accuracy"),
    "F1 Macro": metrics_a.get("eval_f1_macro"),
    "Training Time (s)": 0.0
})


# --- EXPERIMENT 2: MODEL B (STANDARD FULL FINE-TUNING) ---

print("\n" + "="*60)
print("2. TRAINING MODEL B (FULL FINE-TUNING - Performance Baseline)")
print("="*60)

# FIX: Create a NEW TrainingArguments object for Model B
args_b_dict = base_args_dict.copy()
args_b_dict['output_dir'] = OUTPUT_DIR_B
training_args_b = TrainingArguments(**args_b_dict)

trainer_b = Trainer(
    model=model_b,
    args=training_args_b, 
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

train_start_b = time.time()
trainer_b.train()
train_duration_b = time.time() - train_start_b

print(f"\nSaving final Model B to: {FINAL_MODEL_DIR_B}")
trainer_b.save_model(FINAL_MODEL_DIR_B)
tokenizer.save_pretrained(FINAL_MODEL_DIR_B) # Save tokenizer here
metrics_b = trainer_b.evaluate()

print(f"\nFinal Test Results (Model B): {metrics_b}")
EXPERIMENT_LOG.append({
    "Model": "Model B",
    "Method": "Full Fine-Tuning",
    "Trainable Parameters": model_b.num_parameters(),
    "Accuracy": metrics_b.get("eval_accuracy"),
    "F1 Macro": metrics_b.get("eval_f1_macro"),
    "Training Time (s)": train_duration_b
})


# --- EXPERIMENT 3: MODEL C (LoRA/PEFT FINE-TUNING) ---

print("\n" + "="*60)
print("3. TRAINING MODEL C (LoRA/PEFT - Efficiency Baseline)")
print("="*60)

# FIX: Create a NEW TrainingArguments object for Model C
args_c_dict = base_args_dict.copy()
args_c_dict['output_dir'] = OUTPUT_DIR_C
training_args_c = TrainingArguments(**args_c_dict)

trainer_c = Trainer(
    model=model_c, # Use the PEFT-wrapped model
    args=training_args_c, 
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

train_start_c = time.time()
trainer_c.train()
train_duration_c = time.time() - train_start_c

print(f"\nSaving final Model C LoRA adapters to: {FINAL_MODEL_DIR_C}")
# IMPORTANT: Use model.save_pretrained for PEFT models to save only adapters
model_c.save_pretrained(FINAL_MODEL_DIR_C)

# FIX: You MUST save the tokenizer to the LoRA directory for the evaluation notebook to load it!
tokenizer.save_pretrained(FINAL_MODEL_DIR_C) 

metrics_c = trainer_c.evaluate()
print(f"\nFinal Test Results (Model C): {metrics_c}")
EXPERIMENT_LOG.append({
    "Model": "Model C",
    "Method": "LoRA/PEFT Fine-Tuning",
    "Trainable Parameters": model_c.num_parameters(),
    "Accuracy": metrics_c.get("eval_accuracy"),
    "F1 Macro": metrics_c.get("eval_f1_macro"),
    "Training Time (s)": train_duration_c
})

print(f"\n🎉 ALL EXPERIMENTS COMPLETE in {time.time() - start_time:.2f} seconds!")


1. EVALUATING MODEL A (ZERO-SHOT BASELINE)



Final Test Results (Model A): {'eval_loss': 1.391479730606079, 'eval_model_preparation_time': 0.0011, 'eval_accuracy': 0.206, 'eval_f1_macro': 0.10438436526085237, 'eval_runtime': 61.949, 'eval_samples_per_second': 8.071, 'eval_steps_per_second': 0.517}

2. TRAINING MODEL B (FULL FINE-TUNING - Performance Baseline)


c:\Users\joaqu\AI-News-Classification\.venv\Lib\site-packages\transformers\training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
C:\Users\joaqu\AppData\Local\Temp\ipykernel_15036\1764098576.py:54: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_b = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.389100,0.411837,0.870000,0.871049
2,0.304000,0.395886,0.876000,0.876695
3,0.248400,0.388420,0.886000,0.886621



Saving final Model B to: ../models/Distilbert/distilbert-B-final



Final Test Results (Model B): {'eval_loss': 0.3884198069572449, 'eval_accuracy': 0.886, 'eval_f1_macro': 0.8866207032120901, 'eval_runtime': 81.8824, 'eval_samples_per_second': 6.106, 'eval_steps_per_second': 0.391, 'epoch': 3.0}

3. TRAINING MODEL C (LoRA/PEFT - Efficiency Baseline)


c:\Users\joaqu\AI-News-Classification\.venv\Lib\site-packages\transformers\training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
C:\Users\joaqu\AppData\Local\Temp\ipykernel_15036\1764098576.py:94: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_c = Trainer(


Epoch,Training Loss,Validation Loss
